<a href="https://colab.research.google.com/github/raki-rankawat/thesis-v1/blob/master/Model_KD_Combined.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model_KD_Combined

**Knowledge Distillation — 1 Teacher, 2 Students, 2 Init Strategies**

| | Teacher | Student A | Student B |
|---|---|---|---|
| Architecture | `VGG_Pretrained` | `VWW_MobileNetV2` | `VWW_MobileNetV3` |
| Checkpoint | `vgg_pretrained_seed_85.pth` | `mobilenetv2_seed_74.pth` | `mobilenetv3_seed_74.pth` |

**Four training runs produced:**

| Run | Student | Init | Output checkpoint |
|---|---|---|---|
| 1 | MobileNetV2 | FT (warm-start) | `mobilenetv2_kd_ft.pth` |
| 2 | MobileNetV2 | Scratch | `mobilenetv2_kd_scratch.pth` |
| 3 | MobileNetV3 | FT (warm-start) | `mobilenetv3_kd_ft.pth` |
| 4 | MobileNetV3 | Scratch | `mobilenetv3_kd_scratch.pth` |

**Prerequisites:** Run `Model_VGG_Pretrained`, `Model_MobileNetV2`, `Model_MobileNetV3` first.

In [1]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")

Mounted at /content/drive
✅ utils loaded from Drive


In [2]:
# ── Imports ─────────────────────────────────────────────────────────
import os, time
import torch

from utils.dataset import prepare_dataset, get_loaders
from utils.models  import VGG_Pretrained, VWW_MobileNetV2, VWW_MobileNetV3
from utils.train   import setup_device, evaluate, train_kd

device = setup_device(seed=41)

Device: cuda


In [3]:
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")

1/4 Download
⬇️  Downloading VWW archive...
✅ Download complete: /content/vww_work/vw_coco2014_96.tar.gz
2/4 Extract
📦 Extracting VWW archive...
✅ Extraction complete: /content/vww_work/extracted
3/4 Find root
   Root: /content/vww_work/extracted/vw_coco2014_96
4/4 Manifests
✅ Manifests already exist: /content/drive/My Drive/vww_fixed_split_manifests
Train: 7000 | Val: 1500 | Batch: 64


In [4]:
# ── Config ──────────────────────────────────────────────────────────
SAVE_DIR = "/content/drive/My Drive/stm32-thesis/checkpoints"

# ── Checkpoints ──────────────────────────────────────────────────────
TEACHER_CKPT   = f"{SAVE_DIR}/vgg_pretrained_seed_85.pth"
MV2_CKPT       = f"{SAVE_DIR}/mobilenetv2_seed_74.pth"
MV3_CKPT       = f"{SAVE_DIR}/mobilenetv3_seed_74.pth"

# ── Output paths ─────────────────────────────────────────────────────
MV2_KD_FT_PATH     = f"{SAVE_DIR}/mobilenetv2_kd_ft.pth"
MV2_KD_SCR_PATH    = f"{SAVE_DIR}/mobilenetv2_kd_scratch.pth"
MV3_KD_FT_PATH     = f"{SAVE_DIR}/mobilenetv3_kd_ft.pth"
MV3_KD_SCR_PATH    = f"{SAVE_DIR}/mobilenetv3_kd_scratch.pth"

# ── KD hyperparameters ────────────────────────────────────────────────
KD_TEMPERATURE = 4.0
KD_ALPHA       = 0.7
KD_EPOCHS      = 30
KD_LR_FT      = 3e-4   # lower LR for warm-start: preserve existing weights
KD_LR_SCR     = 1e-3   # standard LR for scratch
KD_WD         = 1e-4
KD_PATIENCE   = 10

In [5]:
# ── Load teacher (frozen for all runs) ──────────────────────────────
teacher = VGG_Pretrained().to(device)
teacher.load_state_dict(torch.load(TEACHER_CKPT, map_location=device))
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

teacher_acc = evaluate(teacher, val_loader, device)
print(f"✅ Teacher loaded  |  Val acc: {teacher_acc*100:.2f}%")

Downloading: "https://download.pytorch.org/models/vgg16_bn-6c64b313.pth" to /root/.cache/torch/hub/checkpoints/vgg16_bn-6c64b313.pth


100%|██████████| 528M/528M [00:05<00:00, 107MB/s]


✅ Teacher loaded  |  Val acc: 89.07%


---
## Run 1 — MobileNetV2 · KD Fine-Tune
Warm-start from the `mobilenetv2_seed_74` baseline checkpoint, then distil.

In [6]:
# ── MobileNetV2 · KD FT ─────────────────────────────────────────────
student_mv2_ft = VWW_MobileNetV2().to(device)
student_mv2_ft.load_state_dict(torch.load(MV2_CKPT, map_location=device))
baseline_mv2   = evaluate(student_mv2_ft, val_loader, device)
print(f"MobileNetV2 baseline : {baseline_mv2*100:.2f}%")
if teacher_acc <= baseline_mv2:
    print("⚠️  Teacher not stronger than student — KD may not help")

mv2_ft_acc, mv2_ft_time = train_kd(
    student      = student_mv2_ft,
    teacher      = teacher,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    epochs       = KD_EPOCHS,
    lr           = KD_LR_FT,
    weight_decay = KD_WD,
    temperature  = KD_TEMPERATURE,
    alpha        = KD_ALPHA,
    patience     = KD_PATIENCE,
    save_path    = MV2_KD_FT_PATH,
)

MobileNetV2 baseline : 78.40%
Initial student accuracy: 78.40%
Epoch   1/30 | Train 80.57% | Val 78.40%
Epoch   2/30 | Train 81.20% | Val 78.33%
Epoch   3/30 | Train 81.00% | Val 78.27%
Epoch   4/30 | Train 81.13% | Val 78.53% ✅
Epoch   5/30 | Train 80.84% | Val 77.60%
Epoch   6/30 | Train 81.56% | Val 79.20% ✅
Epoch   7/30 | Train 82.07% | Val 77.13%
Epoch   8/30 | Train 81.86% | Val 78.00%
Epoch   9/30 | Train 82.29% | Val 77.40%
Epoch  10/30 | Train 82.71% | Val 78.27%
Epoch  11/30 | Train 82.84% | Val 77.67%
Epoch  12/30 | Train 82.53% | Val 78.47%
Epoch  13/30 | Train 82.16% | Val 78.53%
Epoch  14/30 | Train 82.24% | Val 78.20%
Epoch  15/30 | Train 82.73% | Val 78.67%
Epoch  16/30 | Train 83.23% | Val 78.73%
🛑 Early stopping at epoch 16

✅ Best KD val acc: 79.20%  (4.6 min)


---
## Run 2 — MobileNetV2 · KD Scratch
Random initialisation — teacher supervision only, no pre-trained weights.

In [7]:
# ── MobileNetV2 · KD Scratch ─────────────────────────────────────────
student_mv2_scr = VWW_MobileNetV2().to(device)
print("✅ MobileNetV2 initialised from scratch")

mv2_scr_acc, mv2_scr_time = train_kd(
    student      = student_mv2_scr,
    teacher      = teacher,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    epochs       = KD_EPOCHS,
    lr           = KD_LR_SCR,
    weight_decay = KD_WD,
    temperature  = KD_TEMPERATURE,
    alpha        = KD_ALPHA,
    patience     = KD_PATIENCE,
    save_path    = MV2_KD_SCR_PATH,
)

✅ MobileNetV2 initialised from scratch
Initial student accuracy: 49.33%
Epoch   1/30 | Train 58.90% | Val 63.27% ✅
Epoch   2/30 | Train 65.43% | Val 64.93% ✅
Epoch   3/30 | Train 67.87% | Val 64.80%
Epoch   4/30 | Train 69.19% | Val 71.60% ✅
Epoch   5/30 | Train 70.09% | Val 70.33%
Epoch   6/30 | Train 71.27% | Val 70.87%
Epoch   7/30 | Train 72.76% | Val 71.67% ✅
Epoch   8/30 | Train 72.57% | Val 72.60% ✅
Epoch   9/30 | Train 73.80% | Val 73.27% ✅
Epoch  10/30 | Train 74.41% | Val 74.53% ✅
Epoch  11/30 | Train 75.00% | Val 76.20% ✅
Epoch  12/30 | Train 74.79% | Val 74.27%
Epoch  13/30 | Train 75.60% | Val 74.87%
Epoch  14/30 | Train 76.27% | Val 74.47%
Epoch  15/30 | Train 75.79% | Val 74.87%
Epoch  16/30 | Train 77.24% | Val 75.27%
Epoch  17/30 | Train 77.20% | Val 75.07%
Epoch  18/30 | Train 77.11% | Val 76.27% ✅
Epoch  19/30 | Train 77.59% | Val 76.60% ✅
Epoch  20/30 | Train 78.51% | Val 76.00%
Epoch  21/30 | Train 79.07% | Val 76.33%
Epoch  22/30 | Train 78.49% | Val 76.47%
Epoch 

---
## Run 3 — MobileNetV3 · KD Fine-Tune
Warm-start from the `mobilenetv3_seed_74` baseline checkpoint, then distil.

In [8]:
# ── MobileNetV3 · KD FT ─────────────────────────────────────────────
student_mv3_ft = VWW_MobileNetV3().to(device)
student_mv3_ft.load_state_dict(torch.load(MV3_CKPT, map_location=device))
baseline_mv3   = evaluate(student_mv3_ft, val_loader, device)
print(f"MobileNetV3 baseline : {baseline_mv3*100:.2f}%")
if teacher_acc <= baseline_mv3:
    print("⚠️  Teacher not stronger than student — KD may not help")

mv3_ft_acc, mv3_ft_time = train_kd(
    student      = student_mv3_ft,
    teacher      = teacher,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    epochs       = KD_EPOCHS,
    lr           = KD_LR_FT,
    weight_decay = KD_WD,
    temperature  = KD_TEMPERATURE,
    alpha        = KD_ALPHA,
    patience     = KD_PATIENCE,
    save_path    = MV3_KD_FT_PATH,
)

MobileNetV3 baseline : 79.13%
Initial student accuracy: 79.13%
Epoch   1/30 | Train 82.01% | Val 78.53%
Epoch   2/30 | Train 82.33% | Val 78.20%
Epoch   3/30 | Train 82.39% | Val 78.33%
Epoch   4/30 | Train 82.39% | Val 77.60%
Epoch   5/30 | Train 82.00% | Val 77.73%
Epoch   6/30 | Train 82.94% | Val 77.33%
Epoch   7/30 | Train 81.99% | Val 76.93%
Epoch   8/30 | Train 82.73% | Val 78.40%
Epoch   9/30 | Train 82.66% | Val 77.67%
Epoch  10/30 | Train 82.47% | Val 77.87%
🛑 Early stopping at epoch 10

✅ Best KD val acc: 79.13%  (2.9 min)


---
## Run 4 — MobileNetV3 · KD Scratch
Random initialisation — teacher supervision only, no pre-trained weights.

In [9]:
# ── MobileNetV3 · KD Scratch ─────────────────────────────────────────
student_mv3_scr = VWW_MobileNetV3().to(device)
print("✅ MobileNetV3 initialised from scratch")

mv3_scr_acc, mv3_scr_time = train_kd(
    student      = student_mv3_scr,
    teacher      = teacher,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    epochs       = KD_EPOCHS,
    lr           = KD_LR_SCR,
    weight_decay = KD_WD,
    temperature  = KD_TEMPERATURE,
    alpha        = KD_ALPHA,
    patience     = KD_PATIENCE,
    save_path    = MV3_KD_SCR_PATH,
)

✅ MobileNetV3 initialised from scratch
Initial student accuracy: 50.00%
Epoch   1/30 | Train 60.61% | Val 61.67% ✅
Epoch   2/30 | Train 65.71% | Val 65.73% ✅
Epoch   3/30 | Train 67.97% | Val 66.27% ✅
Epoch   4/30 | Train 69.80% | Val 69.33% ✅
Epoch   5/30 | Train 71.03% | Val 68.73%
Epoch   6/30 | Train 72.23% | Val 72.73% ✅
Epoch   7/30 | Train 72.60% | Val 71.87%
Epoch   8/30 | Train 73.24% | Val 72.73%
Epoch   9/30 | Train 74.17% | Val 71.80%
Epoch  10/30 | Train 74.47% | Val 74.47% ✅
Epoch  11/30 | Train 74.59% | Val 74.87% ✅
Epoch  12/30 | Train 75.06% | Val 73.40%
Epoch  13/30 | Train 76.13% | Val 73.80%
Epoch  14/30 | Train 75.99% | Val 74.20%
Epoch  15/30 | Train 76.73% | Val 73.47%
Epoch  16/30 | Train 76.73% | Val 74.07%
Epoch  17/30 | Train 77.40% | Val 75.20% ✅
Epoch  18/30 | Train 77.04% | Val 75.40% ✅
Epoch  19/30 | Train 77.40% | Val 75.73% ✅
Epoch  20/30 | Train 78.16% | Val 75.93% ✅
Epoch  21/30 | Train 78.91% | Val 75.33%
Epoch  22/30 | Train 78.59% | Val 75.00%
Epoc

---
## Results Summary

In [10]:
# ── Load best checkpoints and evaluate ─────────────────────────────
def reload_eval(model_cls, ckpt):
    m = model_cls().to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    return evaluate(m, val_loader, device)

final_mv2_ft  = reload_eval(VWW_MobileNetV2, MV2_KD_FT_PATH)
final_mv2_scr = reload_eval(VWW_MobileNetV2, MV2_KD_SCR_PATH)
final_mv3_ft  = reload_eval(VWW_MobileNetV3, MV3_KD_FT_PATH)
final_mv3_scr = reload_eval(VWW_MobileNetV3, MV3_KD_SCR_PATH)

print("=" * 62)
print(f"{'Run':<30} {'Val Acc':>10} {'Time (min)':>12}")
print("-" * 62)
print(f"Teacher (VGG_Pretrained)       {teacher_acc*100:>9.2f}%            —")
print("-" * 62)
print(f"MobileNetV2 baseline           {baseline_mv2*100:>9.2f}%            —")
print(f"MobileNetV2  KD FT             {final_mv2_ft*100:>9.2f}% {mv2_ft_time:>11.1f}")
print(f"MobileNetV2  KD Scratch        {final_mv2_scr*100:>9.2f}% {mv2_scr_time:>11.1f}")
print("-" * 62)
print(f"MobileNetV3 baseline           {baseline_mv3*100:>9.2f}%            —")
print(f"MobileNetV3  KD FT             {final_mv3_ft*100:>9.2f}% {mv3_ft_time:>11.1f}")
print(f"MobileNetV3  KD Scratch        {final_mv3_scr*100:>9.2f}% {mv3_scr_time:>11.1f}")
print("=" * 62)

# Delta over baseline
print()
print("Δ over baseline:")
print(f"  MobileNetV2  FT     : {(final_mv2_ft  - baseline_mv2)*100:+.2f}%")
print(f"  MobileNetV2  Scratch: {(final_mv2_scr - baseline_mv2)*100:+.2f}%")
print(f"  MobileNetV3  FT     : {(final_mv3_ft  - baseline_mv3)*100:+.2f}%")
print(f"  MobileNetV3  Scratch: {(final_mv3_scr - baseline_mv3)*100:+.2f}%")

Run                               Val Acc   Time (min)
--------------------------------------------------------------
Teacher (VGG_Pretrained)           89.07%            —
--------------------------------------------------------------
MobileNetV2 baseline               78.40%            —
MobileNetV2  KD FT                 79.20%         4.6
MobileNetV2  KD Scratch            76.60%         8.3
--------------------------------------------------------------
MobileNetV3 baseline               79.13%            —
MobileNetV3  KD FT                 79.13%         2.9
MobileNetV3  KD Scratch            76.13%         8.6

Δ over baseline:
  MobileNetV2  FT     : +0.80%
  MobileNetV2  Scratch: -1.80%
  MobileNetV3  FT     : +0.00%
  MobileNetV3  Scratch: -3.00%
